# Phase 2: Model Training and Evaluation

In this notebook, we'll train several models, perform hyperparameter tuning, and evaluate their performance based on business tradeoffs.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, f1_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold
import joblib
import os

%matplotlib inline

## 1. Load Preprocessed Data
We'll load the data preprocessed in the previous notebook. For demonstration, we'll repeat the preprocessing logic here.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

data_path = '../data/creditcard.csv'
df = pd.read_csv(data_path)

scaler = StandardScaler()
df['Amount'] = scaler.fit_transform(df[['Amount']])
df['Time'] = scaler.fit_transform(df[['Time']])

X = df.drop('Class', axis=1)
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

## 2. Model Training

### 2.1 Logistic Regression

In [ ]:
lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train_res, y_train_res)
y_pred_lr = lr.predict(X_test)
print("Logistic Regression Classification Report:")
print(classification_report(y_test, y_pred_lr))

### 2.2 Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_res, y_train_res)
y_pred_rf = rf.predict(X_test)
print("Random Forest Classification Report:")
print(classification_report(y_test, y_pred_rf))

### 2.3 XGBoost

In [ ]:
xgb = XGBClassifier(random_state=42, scale_pos_weight=1)
xgb.fit(X_train_res, y_train_res)
y_pred_xgb = xgb.predict(X_test)
print("XGBoost Classification Report:")
print(classification_report(y_test, y_pred_xgb))

## 3. Hyperparameter Tuning (XGBoost)

In [ ]:
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
}

grid_search = GridSearchCV(XGBClassifier(random_state=42), param_grid, scoring='f1', cv=StratifiedKFold(n_splits=3), verbose=1)
grid_search.fit(X_train_res, y_train_res)

best_xgb = grid_search.best_estimator_
print("Best parameters:", grid_search.best_params_)

## 4. Evaluation and Model Selection

### 4.1 Evaluation Metrics
- **Recall is critical:** We want to catch as many fraudulent transactions as possible to minimize financial loss.
- **Precision cannot be ignored:** High false positive rate (FPR) can lead to poor user experience (e.g., blocking legitimate transactions).

In [ ]:
y_pred_best = best_xgb.predict(X_test)
print("Best XGBoost Classification Report:")
print(classification_report(y_test, y_pred_best))
print("ROC-AUC:", roc_auc_score(y_test, y_pred_best))
print("F1-score:", f1_score(y_test, y_pred_best))

### 4.2 Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred_best)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix (Best XGBoost)')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

### 4.3 Save Best Model

In [ ]:
model_path = '../models/best_model.pkl'
joblib.dump(best_xgb, model_path)
print(f"Best model saved to {model_path}")